In [1]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

Found existing installation: openai 1.70.0
Uninstalling openai-1.70.0:
  Successfully uninstalled openai-1.70.0
  Using cached openai-1.70.0-py3-none-any.whl (599 kB)
1.70.0


In [9]:
import openai
import json

def generate_reasoning_tree_json(question, options, model="gpt-4o"):
    """
    Given a clinical question and 4 options, returns a structured reasoning tree JSON from GPT-4o.
    """

    prompt = f"""
You are a clinical reasoning assistant.

A user is asking you to analyze the following USMLE-style clinical question and return a structured JSON file that captures the diagnostic reasoning process.

---

**Question:**
{question}

**Options:**
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

---

**Instructions:**

Your output must be a valid JSON object that represents a diagnostic reasoning tree. 

For each diagnostic option, analyze its strengths and weaknesses **using only the specific information provided in the question prompt**. Do **not** answer based on general knowledge alone. Reference key clinical details from the vignette, including:

- vital signs (e.g., fever, hypotension, tachycardia),
- past medical history (e.g., IV drug use, cancer, immunosuppression),
- physical exam findings (e.g., murmurs, rashes, neurological signs),
- presenting symptoms or contextual clues (e.g., confusion, diarrhea, trauma, location of care, medications).

In addition, populate the `"diagnostic_test"` field as follows:

- If the clinical scenario explicitly **mentions a test**, populate that field accordingly.
- If **no test is mentioned**, but one is clearly implied (e.g., blood cultures for sepsis, ECG for chest pain), **infer** the most relevant diagnostic or monitoring test.
- If there is no reasonable test to mention, you may use `"N/A"` and leave the other fields blank.

---

**JSON Format:**

{{
  "patient_case": {{
  "prompt_question": "Paste the full clinical question text here.",
  "age": "Extract the patient’s age if given (e.g., '42-year-old')",
  "sex": "Extract the patient's sex (e.g., 'male', 'female')",
  "setting": "Mention the care setting if available (e.g., ED, ICU, outpatient, clinical trial).",
  "history": ["List relevant comorbidities, risk factors, or behaviors (e.g., IV drug use, recent surgery, travel)."],
  "symptoms": ["Extract main presenting symptoms (e.g., chest pain, shortness of breath, confusion)."],
  "vitals": ["Extract any vital signs (e.g., temperature, HR, BP, RR)."],
  "physical_exam": ["List key physical findings (e.g., murmur, rash, neuro deficit)."],
  "note": "Summarize the clinical focus of the question — diagnosis, management, or physiology."
  }},
  "diagnostic_test": {{
    "name": "Name the most relevant diagnostic or monitoring procedure if mentioned or clinically implied (e.g., blood culture, CT, echocardiogram, biopsy, telemetry).",
    "procedure": ["Briefly describe how the test would be performed."],
    "stains": ["If applicable (e.g., pathology, cytology), list stains."],
    "findings": ["What would be expected or observed in this patient."]
  }},
  "reasoning_tree": {{
    "start": "Review clinical case and question",
    "branch": [
      {{
        "condition": "Analyze each option based on known pathophysiology, presentation, and clues",
        "result": "Differential reasoning applied",
        "interpretation": "Compare each answer using specific features from the patient’s case",
        "diagnostic_options": [
          {{
            "option": "{options[0]}",
            "description": "...",
            "evidence_for": ["Mention how specific findings support this option."],
            "evidence_against": ["Mention how specific findings argue against it."],
            "final_verdict": "Most likely / Unlikely / Plausible"
          }},
          ...
        ]
      }}
    ],
    "conclusion": "Summarize why the selected answer is best given the case details."
  }}
}}

Respond with only the JSON object — no commentary or explanation outside the JSON.
"""

    try:
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="o3-mini", # gpt-4o
            messages=[{"role": "user", "content": prompt}]
        )
        content = response.choices[0].message.content.strip()
        reasoning_json = json.loads(content)
        return reasoning_json

    except Exception as e:
        print(f"❌ Error processing question: {question[:80]}...\n{e}")
        return None

In [11]:
import os
import json
import time
import pandas as pd
from tqdm.notebook import tqdm

# Load data
df = pd.read_csv("jamao3_randomize_reasoning.csv")

# Create output directory
output_dir = "jama_trees"
os.makedirs(output_dir, exist_ok=True)

# Loop over rows and generate structured JSONs
for i in tqdm(range(len(df))):
    reasoning_text = df.at[i, "o3_mini_reasoning_process_new"]  # updated column
    question_stem = df.at[i, "question"]
    options = {
        "A": df.at[i, "opa"],
        "B": df.at[i, "opb"],
        "C": df.at[i, "opc"],
        "D": df.at[i, "opd"]
    }

    # Combine reasoning text with answer choices to provide more context
    full_reasoning_input = f"""
Question:
{question_stem}

Options:
A. {options['A']}
B. {options['B']}
C. {options['C']}
D. {options['D']}

Reasoning:
{reasoning_text}
    """

    structured = convert_reasoning_to_json(full_reasoning_input)

    # Save each as an individual JSON file
    output_path = os.path.join(output_dir, f"tree_{i}.json")
    with open(output_path, "w") as f:
        json.dump(structured, f, indent=2)

    # Optional delay to be gentle on the API
    time.sleep(0.5)

print("✅ All reasoning trees saved as individual JSON files in 'jama_trees/'")


  0%|          | 0/1034 [00:00<?, ?it/s]

KeyboardInterrupt: 